# Multi-Agent Collaboration

A multi-agent system gives separate model instances bounded tasks and combines their results.


## What to learn

- Delegation: a coordinator asks a worker for a result and keeps control of the task.
- Handoff: control moves to another agent that continues the interaction.
- Parallel workers help when subtasks are independent.
- A result contract states exactly what each worker must return.

## Decision rule

Use one agent by default. Add workers only when context isolation, specialized tools, or genuine parallelism improves the result enough to offset extra cost and coordination failure modes.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
# ALTERNATIVE FRAMEWORK EXAMPLE: OpenAI Agents SDK handoffs
# Import the dependencies used by this example.
import os

if not os.getenv("OPENAI_API_KEY"):
    print("Skipped: set OPENAI_API_KEY to run this example.")
else:
    from agents import Agent, Runner

    billing_agent = Agent(
        name="Billing specialist",
        handoff_description="Handles invoices, refunds, and payment questions.",
        instructions="Resolve billing questions and explain the next action.",
        # Configure the framework object; this line prepares it but may not execute it yet.
        model="gpt-5.6-sol",
    )
    technical_agent = Agent(
        name="Technical specialist",
        handoff_description="Handles errors, crashes, and troubleshooting.",
        instructions="Diagnose technical issues one safe step at a time.",
        model="gpt-5.6-sol",
    )
    triage_agent = Agent(
        name="Triage",
        instructions="Answer simple questions or hand off to the correct specialist.",
        handoffs=[billing_agent, technical_agent],
        model="gpt-5.6-sol",
    )

    result = await Runner.run(triage_agent, "The app crashes when I open an invoice.")
    print("Final agent:", result.last_agent.name)
    print(result.final_output)


In [ ]:
"""Orchestrator-worker vs handoff, both in ~40 lines.

Fake agents demonstrate the two control-flow shapes and the
structured-results rule.
"""

# --- delegation: orchestrator-worker fan-out/fan-in ------------------------
# Define the data shape and small operations before running them.
def worker(mandate):
    """A worker returns a STRUCTURED result — never its 'transcript'."""
    findings = {
        "pricing":  {"finding": "3 tiers, $29-299", "confidence": "high", "sources": 4},
        "reviews":  {"finding": "support complaints trending up", "confidence": "medium", "sources": 9},
        "hiring":   {"finding": "5 open ML roles", "confidence": "high", "sources": 2},
    }
    return {"mandate": mandate["objective"], **findings[mandate["topic"]]}


def orchestrator(task):
    plan = [  # explicit, partitioned mandates — no overlap
        {"topic": "pricing", "objective": "map competitor pricing tiers", "budget_calls": 5},
        {"topic": "reviews", "objective": "summarize customer sentiment", "budget_calls": 8},
        {"topic": "hiring",  "objective": "infer strategy from job posts", "budget_calls": 3},
    ]
    results = [worker(m) for m in plan]        # parallel in production
    high_conf = [r for r in results if r["confidence"] == "high"]
    return {"task": task, "findings": results,
            "synthesis": f"{len(high_conf)}/{len(results)} findings high-confidence"}


report = orchestrator("competitive analysis of AcmeCo")
for f in report["findings"]:
    print(f"  [{f['confidence']:>6}] {f['mandate']}: {f['finding']}")
print(report["synthesis"])

# --- handoff: transfer of control with a structured summary ----------------
def triage_agent(user_msg):
    if "refund" in user_msg:
        summary = {"intent": "refund", "order": "ORD-7", "already_verified": True}
        return {"handoff_to": "billing", "context_summary": summary}
    return {"reply": "How can I help?"}


def billing_agent(context_summary, user_msg):
    v = "verified" if context_summary["already_verified"] else "needs verification"
    return f"Billing here - processing refund for {context_summary['order']} ({v})."


msg = "I want a refund for order ORD-7"
t = triage_agent(msg)
print("\nhandoff:", billing_agent(t["context_summary"], msg))
# Note what crossed the boundary: a compact summary, NOT the transcript.


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
